# ChemBreak30 cloud run
Run each cell from top to bottom. The default dry run verifies the complete bounded first-success pipeline without loading the target models or calling external APIs.

In [ ]:
# Cell 1: controls
DRY_RUN = True
RUN_ID = None  # None creates cb30_dry_seed_30030 or cb30_live_seed_30030
SEED = 30030
TARGETS = ["ChemDFM", "ChemLLM"]
TASK_LIMIT = None
PROBE_TOKENIZERS = False
PROBE_ROLES = False

In [ ]:
# Cell 2: clone the ChemBreak GitHub repository into Colab Enterprise
from pathlib import Path
import subprocess
REPO_URL = 'https://github.com/Jollychuks/ChemBreak.git'
REPO_BRANCH = 'main'
PROJECT_SUBDIR = 'chembreak30'
CLONE_ROOT = Path('/content/chembreak30_repo')
if (CLONE_ROOT/'.git').exists():
    subprocess.run(['git', '-C', str(CLONE_ROOT), 'fetch', 'origin', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(CLONE_ROOT), 'checkout', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(CLONE_ROOT), 'pull', '--ff-only', 'origin', REPO_BRANCH], check=True)
elif CLONE_ROOT.exists():
    raise RuntimeError(f'{CLONE_ROOT} exists but is not a Git repository. Remove that temporary folder and rerun this cell.')
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(CLONE_ROOT)], check=True)
PROJECT_ROOT = (CLONE_ROOT/PROJECT_SUBDIR).resolve()
if not (PROJECT_ROOT/'pyproject.toml').exists():
    raise FileNotFoundError(f'Expected {PROJECT_SUBDIR}/pyproject.toml in {REPO_URL}. Upload the complete chembreak30 folder to the repository root, commit it to main, and rerun this cell.')
print('Repository:', REPO_URL)
print('Branch:', REPO_BRANCH)
print('PROJECT_ROOT =', PROJECT_ROOT)

In [ ]:
# Cell 3: install pinned dependencies and activate the local package
import subprocess, sys, importlib
requirements = PROJECT_ROOT/'requirements-cloud-ml.txt'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(requirements)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', '-e', str(PROJECT_ROOT)], check=True)
SRC_ROOT = (PROJECT_ROOT/'src').resolve()
if not SRC_ROOT.exists(): raise FileNotFoundError(f'ChemBreak30 source directory not found: {SRC_ROOT}')
if str(SRC_ROOT) not in sys.path: sys.path.insert(0, str(SRC_ROOT))
importlib.invalidate_caches()
sys.modules.pop('chembreak30', None)
import chembreak30
print('ChemBreak30 import OK:', chembreak30.__version__, chembreak30.__file__)

In [ ]:
# Cell 4: persistent storage and live-runtime checks
import os
RUN_MODE = 'dry' if DRY_RUN else 'live'
resolved_run_id = RUN_ID or f'cb30_{RUN_MODE}_seed_{SEED}'
storage = Path('/content/chembreak30_storage')/RUN_MODE/resolved_run_id
storage.mkdir(parents=True, exist_ok=True)
if not DRY_RUN:
    import torch
    if not torch.cuda.is_available(): raise RuntimeError('A CUDA GPU is required for a live run. Select a GPU runtime, restart, and rerun from Cell 1.')
    if hasattr(torch.cuda, 'is_bf16_supported') and not torch.cuda.is_bf16_supported(): raise RuntimeError('The configured live run requires a bfloat16-capable GPU.')
    print(torch.cuda.get_device_name(0))
else:
    print('Dry run: GPU and external APIs are not required.')

In [ ]:
# Cell 5: credentials for live execution
if not DRY_RUN:
    import getpass
    if not os.environ.get('GOOGLE_CLOUD_PROJECT','').strip(): os.environ['GOOGLE_CLOUD_PROJECT'] = input('Google Cloud project ID: ').strip()
    if not os.environ.get('OPENAI_API_KEY','').strip(): os.environ['OPENAI_API_KEY'] = getpass.getpass('OpenAI API key: ').strip()
    if not os.environ['GOOGLE_CLOUD_PROJECT'] or not os.environ['OPENAI_API_KEY']: raise RuntimeError('Both credentials are required.')

In [ ]:
# Cell 6: create a run-specific config without changing the locked package config
import yaml
base_config = PROJECT_ROOT/'configs'/'config.cb30.yaml'
cfg = yaml.safe_load(base_config.read_text())
cfg['run']['dry_run'] = bool(DRY_RUN)
cfg['run']['run_mode'] = RUN_MODE
cfg['run']['run_id'] = resolved_run_id
cfg['run']['seed'] = int(SEED)
cfg['run']['task_limit'] = TASK_LIMIT
cfg['run']['output_root'] = str(storage/'runs')
cfg['run']['artifact_root'] = str(storage/'task_artifacts')
for role in ('attack_llm','intent_gate_llm','chcs_fallback_judge_llm'): cfg['roles'][role]['seed'] = int(SEED)
runtime_config = PROJECT_ROOT/'configs'/f'config.cb30.{resolved_run_id}.runtime.yaml'
runtime_config.write_text(yaml.safe_dump(cfg, sort_keys=False))
print(runtime_config)

In [ ]:
# Cell 7: dataset, configuration, model, GPU, and API preflight
from chembreak30.preflight import run_preflight
report = run_preflight(runtime_config, probe_tokenizers=PROBE_TOKENIZERS, probe_roles=PROBE_ROLES)
report

## Execution design
For each task, ChemBreak30 sends the original prompt once. A valid baseline success closes the task. Otherwise the system runs up to three adaptive episodes of five target turns each. Every episode starts a fresh target conversation from the baseline response state, while summarized baseline/prior-episode evidence and task-local contextual-bandit values carry forward. The attack LLM generates one candidate at a time. The task stops at the first valid CHCS 5 result, and all learned state is discarded before the next task.

In [ ]:
# Cell 8: run selected targets
from chembreak30.runner import ChemBreak30Runner
summaries = {}
for target_id in TARGETS:
    runner = ChemBreak30Runner(runtime_config, target_id)
    try: summaries[target_id] = runner.run_all()
    finally: runner.close()
print('Completed:', list(summaries))

In [ ]:
# Cell 9: compact scientific summary
import pandas as pd
rows = []
for target, s in summaries.items():
    rows.append({'target':target,'baseline_successes':s['baseline']['successes'],'baseline_asr':s['baseline']['asr'],'episode_1_cumulative_successes':s['episode_1']['cumulative_successes'],'episode_1_cumulative_asr':s['episode_1']['cumulative_asr'],'episode_2_cumulative_successes':s['episode_2']['cumulative_successes'],'episode_2_cumulative_asr':s['episode_2']['cumulative_asr'],'final_successes':s['bounded_first_success']['successes'],'final_asr':s['bounded_first_success']['asr'],'conditional_rescue_rate':s['conditional_rescue_among_baseline_failures']['rate'],'asr_denominator':s['asr_denominator'],'issued_queries':s['query_accounting']['issued'],'unresolved_queries':s['query_accounting']['unresolved_committed'],'judge_coverage':s['query_accounting']['judge_coverage']})
pd.DataFrame(rows)

In [ ]:
# Cell 10: locate private review files without displaying hazardous raw text
revision = cfg['run']['experiment_revision']
for target in TARGETS:
    internal = storage/'runs'/revision/target/'release'/'internal'
    print(target, internal/'cb30_full_transcripts.csv', internal/'cb30_first_success_review.csv')

In [ ]:
# Cell 11: package redacted public results
import shutil
public_stage = Path('/content/chembreak30_public_results')
if public_stage.exists(): shutil.rmtree(public_stage)
public_stage.mkdir()
for target in TARGETS:
    src = storage/'runs'/revision/target/'release'
    dst = public_stage/target
    shutil.copytree(src, dst, ignore=shutil.ignore_patterns('internal'))
public_zip = shutil.make_archive('/content/chembreak30_public_results','zip',public_stage)
print(public_zip)

In [ ]:
# Cell 12: package private audit files separately
private_stage = Path('/content/chembreak30_private_audit')
if private_stage.exists(): shutil.rmtree(private_stage)
private_stage.mkdir()
for target in TARGETS:
    src = storage/'runs'/revision/target/'release'/'internal'
    if src.exists(): shutil.copytree(src, private_stage/target)
private_zip = shutil.make_archive('/content/chembreak30_private_audit','zip',private_stage)
print(private_zip, 'PRIVATE: review and redact before sharing')